In [1]:
import torch
import einops

In [2]:
def vector_gather(vectors, indices):
    """
    Gathers (batched) vectors according to indices.
    Arguments:
        vectors: Tensor[N, L, D]
        indices: Tensor[N, K] or Tensor[N]
    Returns:
        Tensor[N, K, D] or Tensor[N, D]
    """
    N, L, D = vectors.shape
    squeeze = False
    if indices.ndim == 1:
        squeeze = True
        indices = indices.unsqueeze(-1)
    N2, K = indices.shape
    assert N == N2
    indices = einops.repeat(indices, "N K -> N K D", D=D)
    out = torch.gather(vectors, dim=1, index=indices)
    if squeeze:
        out = out.squeeze(1)
    return out

In [3]:
def dag_loss(targets, transition_matrix, emission_probs, bos_idx=0):
    batch_size, m = targets.shape
    _, l, vocab_size = emission_probs.shape
    dp = torch.zeros((batch_size, m, l))
    bos_emissions = emission_probs[:, 0, bos_idx]
    dp[:, 0, 0] = bos_emissions
    # dp is almost setup correctly, just need to replace every 0 with -inf
    dp[dp == 0] = -float('inf')
    # assumes that transition_matrix and emission_probs are already in log space
    # also we need to tranpose emission_probs so it is vocab_size x l
    # so the vector gather works
    emission_probs = emission_probs.transpose(1, 2)
    for i in range(1, m):
        dp[:, i, :] = vector_gather(emission_probs, targets[:, i]) + (torch.logsumexp(dp[:, i-1, :].unsqueeze(1).transpose(1, 2) + transition_matrix, dim=1))
    return dp

In [ ]:
def fix_probs(probs, mask):
    # assumes probs is already in log space
    # and is a square matrix
    # updates probs so that the sum of each row is 1
    # and any available probability mass is 
    # distributed evenly among the non-masked entries
    batch_size, l, _ = logprobs.shape
    logprobs = logprobs.masked_fill(mask == 1, float('-inf'))
    probsmatrix = torch.exp(logprobs)
    remaining = torch.sum(probsmatrix, dim=2)
    remaining = 1 - remaining
    probnonzero = torch.sum(mask == 0, dim=-1)
    remaining = remaining / probnonzero
    probsmatrix = probsmatrix + remaining.unsqueeze(2)
    probsmatrix = probsmatrix.masked_fill(mask == 1, 0)
    logprobs = torch.log(probsmatrix)
    return logprobs

In [ ]:
vocab_size = 5
example_1_len = 3
factor = 2
num_vertices = example_1_len * factor